Detects spots pairs from STED DNA FISH and confocal RNA FISH images

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from itertools import product
from scipy.optimize import linear_sum_assignment

In [18]:
# detect closes RNA to promoter
def detect_DNA_RNA_spot_pairs(path_spots1, path_spots2, out, ch_p, z_name1, y_name1, x_name1, z_name2, y_name2, x_name2):
    df1 = pd.read_csv(path_spots1)
    df2 = pd.read_csv(path_spots2)
    
    # Get coordinates to match
    spot_coords_ch1 = df1.loc[df1[f'channel_{ch_p}'] == ch_p, [z_name1, y_name1, x_name1]].drop_duplicates(subset=[z_name1, y_name1, x_name1]).values    
    spot_coords_ch2 = df2[[z_name2, y_name2, x_name2]].values
    
    result = defaultdict(list)
    
    # Find the closest spot in df2 for each spot in df1
    for i1, c1 in enumerate(spot_coords_ch1):
        # Calculate distances from spot c1 to all spots in spot_coords_ch2
        distances = np.linalg.norm(spot_coords_ch2 - c1, axis=1)
        # Find the index of the minimum distance
        min_index = np.argmin(distances)
        min_distance = distances[min_index]
        closest_spot = spot_coords_ch2[min_index]
    
        # Store the results
        result['distance_um'].append(min_distance)
        for dim_i, dim in enumerate('zyx'):
            result[f'{dim}_1'].append(c1[dim_i])
            result[f'{dim}_2'].append(closest_spot[dim_i])
    
    result_df = pd.DataFrame(result)
    
    # add acquisition info to df and reshape
    right1 = [z_name1, y_name1, x_name1] 
    right2 = [z_name2, y_name2, x_name2] 
    left1 =  ["z_1","y_1","x_1"] 
    left2 = ["z_2","y_2","x_2"] 
    
    result_df1 = result_df.merge(df1, left_on=left1, right_on=right1 ,how='left')
    result_df2 = result_df1.merge(df2[["img","intensity",z_name2, y_name2, x_name2]], 
                                  left_on=left2, right_on=right2 ,how='left',suffixes=('.1', '.2'))
    
    result_df2.to_csv(out, index=False)

# Match spots and calculate distances

In [20]:
in_path1 = "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240803_DNAFISH/" #upper level experiment folder
in_path2 = "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/20240802_RNAFISH/"
# channels = [0,1] # which channels to match
ch_p = 1 # whihc is the promoter channel
z_name1, y_name1, x_name1 = f"z_global_um_0", f"y_global_um_0", f"x_global_um_0"# names of global coords
z_name2, y_name2, x_name2 = "z_aligned_um", "y_aligned_um", "x_aligned_um"# names of aligned global coords


# voxel_size=(250, 150, 150) #sizes of zyx [nm]
# rel_spot_path1 = "/detections_confocal/merge_global_coords.csv" #spot file path relative to in_path
# rel_spot_path1 = "/distances_sted.csv" #spot file path relative to in_path
rel_spot_path1 = "/detections_sted_0.015/merge_distances_all_n.csv" #spot file path relative to in_path
rel_spot_path2 = "/detections_confocal/merge_global_coords_aligned.csv" #spot file path relative to in_path
out = "/data/agl_data/NanoFISH/Gabi/GS487_Dppa3_all_mESC_RNA-DNA/distances_conf-sted_all_n.csv"

In [21]:
path_spots1 = f"{in_path1}/{rel_spot_path1}" #DNA
path_spots2 = f"{in_path2}/{rel_spot_path2}" #RNA

detect_DNA_RNA_spot_pairs(path_spots1, path_spots2, out, 
                          ch_p,z_name1, y_name1, x_name1,z_name2, y_name2, x_name2)